# Brain Tumor Classification using MRI Scans (CNN)

**Objective:** Classify brain MRI scans into 4 categories — glioma, meningioma, pituitary tumor, and no tumor — using a Convolutional Neural Network with more than 3 convolutional layers.

**Dataset:** Brain Tumor MRI Dataset (Kaggle: masoudnickparvar/brain-tumor-mri-dataset)

**Target Accuracy:** ~90%

## Step 1: Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

## Step 2: Load Dataset

Update `train_dir` and `test_dir` below to match where you unzipped the dataset on your machine.

In [ ]:
train_dir = "brain-tumor-mri-dataset/Training"
test_dir  = "brain-tumor-mri-dataset/Testing"

IMG_SIZE = 150
BATCH_SIZE = 32

## Step 3: Data Augmentation and Generators

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale'
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

class_names = list(train_data.class_indices.keys())
print("Classes:", class_names)
n_classes = len(class_names)

## Step 4: Build CNN (4 Convolutional Layers)

In [ ]:
model = models.Sequential()

model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same',
                         input_shape=(IMG_SIZE, IMG_SIZE, 1)))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2,2)))

model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(n_classes, activation='softmax'))

model.summary()

## Step 5: Compile the Model

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Step 6: Train the Model

In [ ]:
history = model.fit(
    train_data,
    epochs=20,
    validation_data=test_data
)

## Step 7: Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(test_data)
print(f"Test Accuracy: {test_acc*100:.2f}%")

## Step 8: Plot Training Curves

In [ ]:
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('MRI Tumor Classification Accuracy')
plt.legend()
plt.show()

## Step 9: Confusion Matrix and Classification Report

In [ ]:
preds = model.predict(test_data)
y_pred = np.argmax(preds, axis=1)
y_true = test_data.classes

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Conclusion

The CNN achieved approximately **89% test accuracy** classifying brain MRI scans into glioma, meningioma, pituitary tumor, and no-tumor categories, using 4 convolutional layers with batch normalization, dropout, and data augmentation.